# Lab 01 — Kafka & Spark: MovieLens Data Analysis

**Course:** CO3137 — Big Data  
**Semester:** HK252  

---

## Objectives

1. Ingest data from a 3-broker Kafka cluster and resolve binary-format issues.  
2. Identify the **top-5 movies** by average rating (with rating count > 30).  
3. Determine the **5 worst tags** associated with the lowest average ratings.  
4. Perform deeper analysis on whether those tags genuinely correlate with lower movie ratings.

---
## 1. Environment Setup & Spark Initialization

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, sum as _sum, avg, count,
    round as _round, desc, asc, collect_list, size
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType,
    StringType, LongType, DoubleType
)

spark = (
    SparkSession.builder
    .appName("Lab01_MovieLens_Kafka")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0,"
        "org.apache.kafka:kafka-clients:3.6.0,"
        "org.apache.spark:spark-streaming-kafka-0-10_2.13:4.0.0"
    )
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

Spark version : 4.0.0
Spark UI      : http://host.docker.internal:4040


---
## 2. Kafka Data Preparation

Download the MovieLens dataset from Kaggle, create Kafka topics on the 3-broker cluster, and push the data into Kafka.

In [2]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic

# ----- Download dataset -----
path = kagglehub.dataset_download("grouplens/movielens-latest-small")
print(f"Dataset path: {path}")

df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies  = spark.read.csv(path + "/movies.csv",  header=True, inferSchema=True)
df_tags    = spark.read.csv(path + "/tags.csv",    header=True, inferSchema=True)

print("=== Ratings ===")
df_ratings.show(3, truncate=False)
print("=== Movies ===")
df_movies.show(3, truncate=False)
print("=== Tags ===")
df_tags.show(3, truncate=False)

C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|████████████████████████████████████████████████████████████████████████████████| 971k/971k [00:01<00:00, 847kB/s]

Extracting files...


Dataset path: C:\Users\ASUS\.cache\kagglehub\datasets\grouplens\movielens-latest-small\versions\2
=== Ratings ===
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
+------+-------+------+---------+
only showing top 3 rows
=== Movies ===
+-------+-----------------------+-------------------------------------------+
|movieId|title                  |genres                                     |
+-------+-----------------------+-------------------------------------------+
|1      |Toy Story (1995)       |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)         |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)|Comedy|Romance                             |
+-------+-----------------------+-------------------------------------------+
only showing top 3 rows
=== Tags ===
+------+-------+----------

In [3]:
# ----- Kafka broker configuration -----
KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"

admin_client = AdminClient({"bootstrap.servers": KAFKA_BROKERS})

# Delete existing topics (ignore errors if they don't exist)
existing = admin_client.list_topics(timeout=10).topics.keys()
topics_to_delete = [t for t in ["ratings", "movies", "tags"] if t in existing]
if topics_to_delete:
    admin_client.delete_topics(topics_to_delete, operation_timeout=10)
    import time; time.sleep(2)  # wait for deletion to propagate
    print(f"Deleted topics: {topics_to_delete}")

# Create topics with replication factor = 3 (one replica per broker)
new_topics = [
    NewTopic(topic="ratings", num_partitions=1, replication_factor=3),
    NewTopic(topic="movies",  num_partitions=1, replication_factor=3),
    NewTopic(topic="tags",    num_partitions=1, replication_factor=3),
]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' created successfully.")
    except Exception as e:
        print(f"Failed to create topic '{topic}': {e}")

Topic 'ratings' created successfully.
Topic 'movies' created successfully.
Topic 'tags' created successfully.


In [4]:
# ----- Push DataFrames to Kafka -----
for name, df_src in [("ratings", df_ratings), ("movies", df_movies), ("tags", df_tags)]:
    df_src.selectExpr("to_json(struct(*)) AS value") \
        .write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BROKERS) \
        .option("topic", name) \
        .save()
    print(f"Topic '{name}': {df_src.count()} records pushed.")

print("\nAll data successfully loaded into Kafka.")

Topic 'ratings': 100836 records pushed.
Topic 'movies': 9742 records pushed.
Topic 'tags': 3683 records pushed.

All data successfully loaded into Kafka.


---
## 3. Exercise 1 — Read Data from Kafka & Resolve Format Issues

When reading from Kafka, the `value` column is stored as **binary** (`bytes`).  
We must:
1. Cast the binary `value` to a UTF-8 **string** (JSON).  
2. Parse the JSON string into a proper Spark **struct** using a schema.  
3. Expand the struct fields into individual DataFrame columns.

In [5]:
# ----- Step 1: Read raw data from Kafka topics -----
def read_kafka_topic(topic: str) -> "DataFrame":
    """Read a Kafka topic and return the raw DataFrame."""
    return (
        spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKERS)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
    )

raw_ratings = read_kafka_topic("ratings")
raw_movies  = read_kafka_topic("movies")
raw_tags    = read_kafka_topic("tags")

print("Raw Kafka schema (ratings):")
raw_ratings.printSchema()
print("Raw value sample (binary format — this is the 'error'):")
raw_ratings.select("value").show(3, truncate=False)

Raw Kafka schema (ratings):
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

Raw value sample (binary format — this is the 'error'):
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                             |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[7B 22 75 73 65 72 49 64 22 3A 31 2C 22 6D 6F 76 69 65 49 64 22 3A 31 2C 22 72 61 74 69 6

In [6]:
# ----- Step 2: Cast binary → string, then parse JSON with explicit schemas -----

# Define schemas matching the original CSV structure
ratings_schema = StructType([
    StructField("userId",    IntegerType(), True),
    StructField("movieId",   IntegerType(), True),
    StructField("rating",    DoubleType(),  True),
    StructField("timestamp", LongType(),    True),
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title",   StringType(),  True),
    StructField("genres",  StringType(),  True),
])

tags_schema = StructType([
    StructField("userId",    IntegerType(), True),
    StructField("movieId",   IntegerType(), True),
    StructField("tag",       StringType(),  True),
    StructField("timestamp", LongType(),    True),
])

def parse_kafka_topic(raw_df, schema):
    """
    Resolve Kafka binary format:
      1. CAST(value AS STRING) — convert binary to JSON string
      2. from_json(...)        — parse JSON string into struct columns
      3. select("data.*")      — flatten struct into top-level columns
    """
    return (
        raw_df
        .selectExpr("CAST(value AS STRING) AS json_str")
        .select(from_json(col("json_str"), schema).alias("data"))
        .select("data.*")
    )

ratings = parse_kafka_topic(raw_ratings, ratings_schema)
movies  = parse_kafka_topic(raw_movies,  movies_schema)
tags    = parse_kafka_topic(raw_tags,     tags_schema)

print("=== Parsed Ratings (from Kafka) ===")
ratings.show(5, truncate=False)
ratings.printSchema()

print("=== Parsed Movies (from Kafka) ===")
movies.show(5, truncate=False)

print("=== Parsed Tags (from Kafka) ===")
tags.show(5, truncate=False)

print(f"Ratings count : {ratings.count()}")
print(f"Movies count  : {movies.count()}")
print(f"Tags count    : {tags.count()}")

=== Parsed Ratings (from Kafka) ===
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
+------+-------+------+---------+
only showing top 5 rows
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)

=== Parsed Movies (from Kafka) ===
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy           

**Explanation of the format error and resolution:**

Kafka stores message values as **raw bytes** (binary). When we pushed DataFrames into Kafka using `to_json(struct(*))`, the data was serialized as JSON strings, but Kafka still stores and returns them as binary.

Reading them directly (without casting) results in unreadable byte arrays. The solution is a 3-step pipeline:

| Step | Operation | Purpose |
|------|-----------|--------|
| 1 | `CAST(value AS STRING)` | Decode binary bytes → UTF-8 JSON string |
| 2 | `from_json(col, schema)` | Parse the JSON string into a typed Spark struct |
| 3 | `select("data.*")` | Flatten the struct into individual columns |

---
## 4. Exercise 2 — Top 5 Movies by Average Rating (rating count > 30)

In [7]:
# Aggregate ratings per movie
movie_stats = (
    ratings
    .groupBy("movieId")
    .agg(
        _round(avg("rating"), 2).alias("avg_rating"),
        count("rating").alias("rating_count")
    )
    .filter(col("rating_count") > 30)
)

# Join with movie titles and sort
top5_movies = (
    movie_stats
    .join(movies, on="movieId", how="inner")
    .select("movieId", "title", "genres", "avg_rating", "rating_count")
    .orderBy(desc("avg_rating"), desc("rating_count"))
    .limit(5)
)

print("=" * 80)
print("TOP 5 MOVIES BY AVERAGE RATING  (rating count > 30)")
print("=" * 80)
top5_movies.show(truncate=False)

TOP 5 MOVIES BY AVERAGE RATING  (rating count > 30)
+-------+---------------------------------------------------------------------------+---------------------------+----------+------------+
|movieId|title                                                                      |genres                     |avg_rating|rating_count|
+-------+---------------------------------------------------------------------------+---------------------------+----------+------------+
|318    |Shawshank Redemption, The (1994)                                           |Crime|Drama                |4.43      |317         |
|1204   |Lawrence of Arabia (1962)                                                  |Adventure|Drama|War        |4.3       |45          |
|858    |Godfather, The (1972)                                                      |Crime|Drama                |4.29      |192         |
|2959   |Fight Club (1999)                                                          |Action|Crime|Drama|Thriller|4.27   

---
## 5. Exercise 3 — 5 Worst Tags (Lowest Average Rating)

In [8]:
# Join tags with ratings on movieId to get the rating associated with each tag
tags_with_ratings = tags.join(ratings, on="movieId", how="inner")

# Normalize tag text to lowercase for consistent grouping
from pyspark.sql.functions import lower, trim

worst_tags = (
    tags_with_ratings
    .withColumn("tag_clean", lower(trim(col("tag"))))
    .groupBy("tag_clean")
    .agg(
        _round(avg("rating"), 2).alias("avg_rating"),
        count("rating").alias("rating_count")
    )
    .orderBy(asc("avg_rating"))
    .limit(5)
)

print("=" * 80)
print("5 WORST TAGS  (associated with lowest average rating)")
print("=" * 80)
worst_tags.show(truncate=False)

5 WORST TAGS  (associated with lowest average rating)
+---------+----------+------------+
|tag_clean|avg_rating|rating_count|
+---------+----------+------------+
|symbolic |0.5       |1           |
|stage    |1.75      |2           |
|tokyo    |2.0       |1           |
|snl      |2.1       |10          |
|jungle   |2.14      |11          |
+---------+----------+------------+



---
## 6. Exercise 4 — Deep Analysis: Do These Tags Truly Indicate Lower Ratings?

The previous result shows tags *associated* with low average ratings, but does that mean movies with these tags generally receive lower ratings? We investigate by:

1. For each of the 5 worst tags, finding all movies that carry that tag.  
2. Computing the **per-movie average rating** for those movies.  
3. Counting how many distinct movies carry each tag.  
4. Comparing the per-tag movie average with the **global average rating** across the entire dataset.

In [9]:
# Collect the 5 worst tag names
worst_tag_names = [row["tag_clean"] for row in worst_tags.collect()]
print(f"Worst tags to analyse: {worst_tag_names}\n")

# Global average rating as a baseline
global_avg = ratings.agg(_round(avg("rating"), 2)).collect()[0][0]
print(f"Global average rating (all movies): {global_avg}\n")

Worst tags to analyse: ['symbolic', 'stage', 'tokyo', 'snl', 'jungle']

Global average rating (all movies): 3.5



In [10]:
# For each worst tag: find associated movies, their individual avg ratings, and movie count
from pyspark.sql.functions import countDistinct

# Prepare: tags with cleaned text joined with ratings
tags_clean = tags.withColumn("tag_clean", lower(trim(col("tag"))))

# Filter only movies that have any of the 5 worst tags
worst_tag_movies = (
    tags_clean
    .filter(col("tag_clean").isin(worst_tag_names))
    .select("tag_clean", "movieId")
    .distinct()
)

# Join with ratings to compute per-movie average rating for these tagged movies
worst_tag_movie_ratings = (
    worst_tag_movies
    .join(ratings, on="movieId", how="inner")
    .groupBy("tag_clean", "movieId")
    .agg(_round(avg("rating"), 2).alias("movie_avg_rating"))
)

# Join with movie titles for readability
worst_tag_movie_detail = (
    worst_tag_movie_ratings
    .join(movies, on="movieId", how="inner")
    .select("tag_clean", "movieId", "title", "movie_avg_rating")
    .orderBy("tag_clean", asc("movie_avg_rating"))
)

print("=" * 80)
print("PER-MOVIE AVERAGE RATINGS FOR MOVIES WITH THE 5 WORST TAGS")
print("=" * 80)
worst_tag_movie_detail.show(50, truncate=False)

PER-MOVIE AVERAGE RATINGS FOR MOVIES WITH THE 5 WORST TAGS
+---------+-------+---------------------------------------------+----------------+
|tag_clean|movieId|title                                        |movie_avg_rating|
+---------+-------+---------------------------------------------+----------------+
|jungle   |1474   |Jungle2Jungle (a.k.a. Jungle 2 Jungle) (1997)|2.14            |
|snl      |2296   |Night at the Roxbury, A (1998)               |2.1             |
|stage    |8943   |Being Julia (2004)                           |1.75            |
|symbolic |26717  |Begotten (1990)                              |0.5             |
|tokyo    |6407   |Walk, Don't Run (1966)                       |2.0             |
+---------+-------+---------------------------------------------+----------------+



In [11]:
# Summary statistics per tag
tag_summary = (
    worst_tag_movie_ratings
    .groupBy("tag_clean")
    .agg(
        countDistinct("movieId").alias("num_movies"),
        _round(avg("movie_avg_rating"), 2).alias("mean_of_movie_avgs"),
    )
    .orderBy(asc("mean_of_movie_avgs"))
)

print("=" * 80)
print("SUMMARY: WORST TAGS vs GLOBAL AVERAGE")
print(f"Global average rating: {global_avg}")
print("=" * 80)
tag_summary.show(truncate=False)

SUMMARY: WORST TAGS vs GLOBAL AVERAGE
Global average rating: 3.5
+---------+----------+------------------+
|tag_clean|num_movies|mean_of_movie_avgs|
+---------+----------+------------------+
|symbolic |1         |0.5               |
|stage    |1         |1.75              |
|tokyo    |1         |2.0               |
|snl      |1         |2.1               |
|jungle   |1         |2.14              |
+---------+----------+------------------+



### Interpretation

From the results above we can draw conclusions:

- **If `mean_of_movie_avgs` < global average (`~3.5`):** Movies carrying this tag do tend to receive lower ratings overall, suggesting the tag reflects a genuinely less-appreciated quality (e.g., "boring", "bad acting").

- **If `mean_of_movie_avgs` ≈ or > global average:** The low tag-level average is likely driven by a **small number of low-rated reviews** rather than the movies themselves being poorly received. The tag may appear on otherwise well-rated movies, meaning the tag itself is *not* a reliable indicator of movie quality.

- **`num_movies` (tag frequency):** Tags appearing on very few movies (1–2) are unreliable signals — their averages are heavily influenced by individual outliers.

**Conclusion:** A low tag-level average rating does not necessarily mean that movies associated with that tag are bad. The relationship depends on sample size and whether the low ratings are consistent across multiple movies or driven by outliers.

---
## 7. Cleanup

In [ ]:
spark.stop()
print("SparkSession stopped.")